[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/03_tensors_the_real_metal.ipynb)

# ⚒️ Act I · Quest 03 — Tensors — The Real Metal

*Your engine used one number at a time. Tensors do millions at once — here's the toolkit.*

⬅️ [02_build_your_own_autograd](02_build_your_own_autograd.ipynb)  •  [04_autograd_unmasked](04_autograd_unmasked.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## Why your `Value` engine can't scale

Training XOR took 4 examples × 17 knobs. MNIST needs 60,000 images × ~100k knobs. Looping over
Python scalars would take *days*. A **tensor** is an n-dimensional array whose operations run in
optimized C/CUDA — the same math, thousands of times faster. Watch:

In [ ]:
import time

# dot product of 1M numbers: pure Python vs tensor
a_list = [random.random() for _ in range(1_000_000)]
b_list = [random.random() for _ in range(1_000_000)]

t0 = time.time()
s = sum(x * y for x, y in zip(a_list, b_list))
py_ms = (time.time() - t0) * 1000

a_t, b_t = torch.tensor(a_list), torch.tensor(b_list)
t0 = time.time()
s_t = a_t @ b_t
torch_ms = (time.time() - t0) * 1000

print(f"pure Python: {py_ms:7.1f} ms")
print(f"tensor     : {torch_ms:7.2f} ms   ({py_ms / max(torch_ms, 1e-6):.0f}x faster, same answer: {abs(s - s_t.item()) < 1e-3})")

### Creating tensors & the big three properties: `shape`, `dtype`, `device`

In [ ]:
t = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print("shape:", t.shape, "| dtype:", t.dtype, "| device:", t.device)

print("\nzeros:", torch.zeros(2, 3).shape)
print("random:", torch.randn(2, 3).shape)          # normal distribution
print("range:", torch.arange(0, 10, 2))
print("evenly spaced:", torch.linspace(0, 1, 5))
print("int -> float:", torch.arange(3).float().dtype)
# .to(device) moves tensors to the GPU when you have one (Colab!)
print("on device:", torch.randn(2).to(device).device)

### Indexing & slicing — an image is just a tensor

In [ ]:
# Build a tiny "image" and manipulate it with pure indexing
img = torch.zeros(8, 8)
img[2:6, 2:6] = 1.0                     # a bright square

fig, ax = plt.subplots(1, 4, figsize=(10, 2.5))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("original")
ax[1].imshow(img.flip(1), cmap="gray"); ax[1].set_title("h-flip")
ax[2].imshow(img[::2, ::2], cmap="gray"); ax[2].set_title("2x downsample")
ax[3].imshow(img.T.roll(2, dims=0), cmap="gray"); ax[3].set_title("transpose+roll")
for a in ax: a.axis("off")
plt.show()

print("row 3:", img[3])
print("bright pixels:", (img > 0.5).sum().item())

### Broadcasting — the superpower

Operating on mismatched shapes without loops. Rule: align shapes **from the right**; each pair
of dims must be equal, or one of them must be `1` (it stretches).

`(8,1) + (1,8) → (8,8)`  ·  `(N,3) - (3,) → (N,3)`

In [ ]:
# A radial gradient image in ONE line of broadcasting — no loops
n = 64
coord = torch.arange(n).float()
dist = ((coord[:, None] - n/2) ** 2 + (coord[None, :] - n/2) ** 2).sqrt()  # (64,1) & (1,64) -> (64,64)
plt.imshow(dist, cmap="magma"); plt.title("distance from center — pure broadcasting"); plt.colorbar(); plt.show()

# The most common real-world use: normalize features per column
data = torch.randn(200, 3) * torch.tensor([1., 10., 100.]) + torch.tensor([0., 5., -50.])
normed = (data - data.mean(dim=0)) / data.std(dim=0)     # (200,3) - (3,) broadcasts
print("means ~0:", normed.mean(0).round(decimals=4), " stds ~1:", normed.std(0).round(decimals=4))

### Reshaping & matmul — the two ops you'll use every single day

In [ ]:
x = torch.arange(24.)
print("as (2,3,4):", x.reshape(2, 3, 4).shape)
print("flatten:", x.reshape(2, 3, 4).reshape(2, -1).shape, " (-1 = 'figure it out')")

batch = torch.randn(32, 1, 20, 20)                    # 32 glyph images
print("flatten batch for a linear layer:", batch.flatten(1).shape)   # keep dim 0

# Matmul IS the neural network: (batch, features) @ (features, out) -> (batch, out)
W = torch.randn(400, 10)
out = batch.flatten(1) @ W
print("layer output:", out.shape)

# Reductions
m = torch.tensor([[1., 5., 3.], [8., 2., 9.]])
print("\nsum all:", m.sum().item(), "| per-column max:", m.max(dim=0).values, "| argmax per row:", m.argmax(dim=1))

> ⚠️ **`view` vs `reshape`**: `view` requires contiguous memory; `reshape` always works. Use
> `reshape` until you have a reason not to. And **`permute` vs `reshape` are NOT the same** —
> `permute` reorders axes (moves data around), `reshape` just re-chops the same order.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda t: torch.is_tensor(t) and t.shape == (3, 4) and (t == 7).all(),
    "torch.full((3, 4), 7.0) — or ones * 7")
_register("bullseye", 15,
    lambda img: torch.is_tensor(img) and img.shape == (32, 32)
                and img[16, 16] == 1 and img[0, 0] == 0
                and abs(img.float().mean().item() - ((((torch.arange(32.)[:,None]-16)**2 + (torch.arange(32.)[None,:]-16)**2).sqrt() <= 8).float().mean().item())) < 1e-6,
    "distance-from-center <= 8 -> 1.0 else 0.0, on a 32x32 grid, centered at (16,16). Broadcasting, no loops!")
_register("normalize", 15,
    lambda f: (lambda z: torch.allclose(z.mean(0), torch.zeros(4), atol=1e-5) and torch.allclose(z.std(0), torch.ones(4), atol=1e-4))(f(torch.randn(100, 4) * 7 + 3)),
    "def normalize(x): return (x - x.mean(0)) / x.std(0)")
_register("shape_oracle", 10,
    lambda s: tuple(s) == (16, 8, 5),
    "A is (16, 8, 32), B is (32, 5). What is (A @ B).shape? Submit a tuple.")

In [ ]:
check("warmup", torch.full((3, 4), 7.0))

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `bullseye` (15 XP) — a 32×32 tensor: 1.0 where distance from center (16,16) ≤ 8, else 0.0. **No loops.**
- `normalize` (15 XP) — write `normalize(x)` that standardizes each column; submit the *function*.
- `shape_oracle` (10 XP) — predict `(A @ B).shape` for `A:(16,8,32)`, `B:(32,5)` — submit a tuple *without running it*.

In [ ]:
# ⚔️ your attempts here...

# xp_report()